# 108 — TCGA Somatic Mutation Acquisition and Audit

## Objective

Acquire, audit, and freeze a single TCGA/GDC somatic-mutation resource for
secondary genomic-context characterization of the already frozen Phase 4
programs.

The selected resource is the open-access GDC `Masked Somatic Mutation` output
generated by the `Aliquot Ensemble Somatic Variant Merging and Masking`
workflow for TCGA primary-tumor whole-exome sequencing samples.

This notebook will establish the mutation-resource provenance, file and case
coverage, sample/case mapping, project composition, residual multiplicity, and
compatibility with the frozen 9,965-case TCGA multi-omic cohort before any
mutation-based biological characterization is performed.

## Scientific role

Somatic mutations are a secondary molecular-context layer.

They are not used to discover, rescue, exclude, reweight, reorient, or rename
the frozen tumor or consensus programs. Mutation associations generated
downstream therefore represent contextual computational evidence rather than a
new discovery modality or causal evidence.

## Frozen acquisition scope

The GDC cohort and file selection was defined using:

- program: `TCGA`;
- tissue type: `Tumor`;
- tumor descriptor: `Primary`;
- data category: `Simple Nucleotide Variation`;
- data type: `Masked Somatic Mutation`;
- workflow type: `Aliquot Ensemble Somatic Variant Merging and Masking`;
- experimental strategy: `WXS`;
- access: `Open`.

The cohort export, manifest, sample sheet, and metadata JSON are retained as
the authoritative acquisition provenance for this resource.

## Main tasks

This notebook will:

1. audit the frozen GDC cohort, manifest, sample sheet, and metadata exports;
2. characterize file, case, sample, project, and workflow composition;
3. evaluate residual file and sample multiplicity per TCGA case;
4. compare mutation-resource coverage with the frozen 9,965-case TCGA
   multi-omic cohort;
5. define a deterministic downstream-compatible mutation handoff without
   arbitrary post-hoc case rescue;
6. download the selected open-access mutation files only after the frozen
   manifest has passed the acquisition audit;
7. validate downloaded file identity against the GDC manifest; and
8. persist reproducible case/file mappings, coverage summaries, and analysis
   metadata for notebook 450.

## Scope boundaries

This notebook will not:

- combine mutation calls from different callers or processing workflows;
- perform mutation–program association testing;
- define cancer-driver genes or pathways from the observed results;
- calculate tumor mutation burden unless a comparable definition is
  prespecified downstream;
- analyze copy-number alterations;
- alter any frozen Phase 2–4 program representation;
- infer biological causality; or
- assign clinical resistance or treatment-response labels.

## Downstream handoff

The frozen output of notebook 108 will be consumed by:

`450 — Secondary Genomic Context Characterization`

Notebook 450 will perform lineage-aware secondary characterization using the
already frozen program representations. Negative, lineage-specific,
heterogeneous, sparse, or non-recurrent mutation associations will remain valid
outcomes.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import hashlib
import json

import pandas as pd

from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import Paths, project_relative_path

In [2]:
# =============================================================================
# Authoritative GDC mutation input paths
# =============================================================================

MUTATION_MANIFEST_DIR = (
    Paths.config
    / "manifests"
    / "tcga_mutation"
)

COHORT_PATH = (
    MUTATION_MANIFEST_DIR
    / "gdc_cohort_tcga_primary_tumor_wxs_masked_somatic_mutation.tsv"
)

MANIFEST_PATH = (
    MUTATION_MANIFEST_DIR
    / "gdc_manifest_tcga_primary_tumor_wxs_masked_somatic_mutation.txt"
)

METADATA_PATH = (
    MUTATION_MANIFEST_DIR
    / "gdc_metadata_tcga_primary_tumor_wxs_masked_somatic_mutation.json"
)

SAMPLE_SHEET_PATH = (
    MUTATION_MANIFEST_DIR
    / "gdc_sample_sheet_tcga_primary_tumor_wxs_masked_somatic_mutation.tsv"
)

In [3]:
# =============================================================================
# Validate authoritative GDC mutation inputs
# =============================================================================

mutation_input_paths = {
    "cohort": COHORT_PATH,
    "manifest": MANIFEST_PATH,
    "metadata": METADATA_PATH,
    "sample_sheet": SAMPLE_SHEET_PATH,
}

missing_inputs = [
    name
    for name, path in mutation_input_paths.items()
    if not path.exists()
]

if missing_inputs:
    raise FileNotFoundError(
        "Missing authoritative mutation inputs: "
        + ", ".join(missing_inputs)
    )

for name, path in mutation_input_paths.items():
    print(
        f"{name}: "
        f"{project_relative_path(path)}"
    )

cohort: config/manifests/tcga_mutation/gdc_cohort_tcga_primary_tumor_wxs_masked_somatic_mutation.tsv
manifest: config/manifests/tcga_mutation/gdc_manifest_tcga_primary_tumor_wxs_masked_somatic_mutation.txt
metadata: config/manifests/tcga_mutation/gdc_metadata_tcga_primary_tumor_wxs_masked_somatic_mutation.json
sample_sheet: config/manifests/tcga_mutation/gdc_sample_sheet_tcga_primary_tumor_wxs_masked_somatic_mutation.tsv


In [4]:
# =============================================================================
# Load authoritative GDC mutation inputs
# =============================================================================

cohort = pd.read_csv(
    COHORT_PATH,
    sep="\t",
)

manifest = pd.read_csv(
    MANIFEST_PATH,
    sep="\t",
)

sample_sheet = pd.read_csv(
    SAMPLE_SHEET_PATH,
    sep="\t",
)

with METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    metadata = json.load(handle)

print(f"Cohort rows:      {len(cohort):,}")
print(f"Manifest rows:    {len(manifest):,}")
print(f"Sample-sheet rows:{len(sample_sheet):,}")
print(f"Metadata records: {len(metadata):,}")

Cohort rows:      10,017
Manifest rows:    10,171
Sample-sheet rows:10,171
Metadata records: 10,171


In [5]:
# =============================================================================
# Inspect GDC export schemas
# =============================================================================

print("Cohort columns:")
print(cohort.columns.tolist())

print("\nManifest columns:")
print(manifest.columns.tolist())

print("\nSample-sheet columns:")
print(sample_sheet.columns.tolist())

print("\nMetadata container:")
print(type(metadata).__name__)

if isinstance(metadata, list) and metadata:
    print("\nMetadata first-record keys:")
    print(sorted(metadata[0].keys()))

Cohort columns:
['id']

Manifest columns:
['id', 'filename', 'md5', 'size', 'state']

Sample-sheet columns:
['File ID', 'File Name', 'Data Category', 'Data Type', 'Project ID', 'Case ID', 'Sample ID', 'Tissue Type', 'Tumor Descriptor', 'Specimen Type', 'Preservation Method']

Metadata container:
list

Metadata first-record keys:
['access', 'analysis', 'annotations', 'associated_entities', 'data_category', 'data_format', 'data_type', 'experimental_strategy', 'file_id', 'file_name', 'file_size', 'md5sum', 'platform', 'state', 'submitter_id']


In [6]:
# =============================================================================
# Inspect nested GDC metadata structure
# =============================================================================

first_metadata_record = metadata[0]

analysis = first_metadata_record.get("analysis")
associated_entities = first_metadata_record.get(
    "associated_entities",
    [],
)

print("Analysis type:")
print(type(analysis).__name__)

if isinstance(analysis, dict):
    print("\nAnalysis keys:")
    print(sorted(analysis.keys()))

print("\nAssociated-entity count:")
print(len(associated_entities))

if associated_entities:
    print("\nFirst associated-entity keys:")
    print(sorted(associated_entities[0].keys()))
    print("\nFirst associated entity:")
    print(associated_entities[0])

Analysis type:
dict

Analysis keys:
['analysis_id', 'created_datetime', 'input_files', 'state', 'submitter_id', 'updated_datetime', 'workflow_link', 'workflow_type', 'workflow_version']

Associated-entity count:
2

First associated-entity keys:
['case_id', 'entity_id', 'entity_submitter_id', 'entity_type']

First associated entity:
{'entity_submitter_id': 'TCGA-38-4631-11A-01D-1753-08', 'entity_type': 'aliquot', 'case_id': '2483621a-4db3-41ab-aa33-b9427ea8a0af', 'entity_id': '84b00131-cbbe-461f-a006-5277e14b05f9'}


In [7]:
# =============================================================================
# Characterize frozen GDC mutation metadata scope
# =============================================================================

metadata_scope = pd.DataFrame(
    {
        "file_id": record["file_id"],
        "workflow_type": record["analysis"]["workflow_type"],
        "experimental_strategy": record["experimental_strategy"],
        "data_type": record["data_type"],
        "data_format": record["data_format"],
        "access": record["access"],
        "n_associated_entities": len(
            record.get("associated_entities", [])
        ),
        "n_associated_cases": len(
            {
                entity["case_id"]
                for entity in record.get("associated_entities", [])
                if entity.get("case_id") is not None
            }
        ),
        "entity_types": "|".join(
            sorted(
                {
                    entity["entity_type"]
                    for entity in record.get(
                        "associated_entities",
                        [],
                    )
                }
            )
        ),
    }
    for record in metadata
)

for column in [
    "workflow_type",
    "experimental_strategy",
    "data_type",
    "data_format",
    "access",
    "n_associated_entities",
    "n_associated_cases",
    "entity_types",
]:
    print(f"\n{column}:")
    print(metadata_scope[column].value_counts(dropna=False))


workflow_type:
workflow_type
Aliquot Ensemble Somatic Variant Merging and Masking    10171
Name: count, dtype: int64

experimental_strategy:
experimental_strategy
WXS    10171
Name: count, dtype: int64

data_type:
data_type
Masked Somatic Mutation    10171
Name: count, dtype: int64

data_format:
data_format
MAF    10171
Name: count, dtype: int64

access:
access
open    10171
Name: count, dtype: int64

n_associated_entities:
n_associated_entities
2    10171
Name: count, dtype: int64

n_associated_cases:
n_associated_cases
1    10171
Name: count, dtype: int64

entity_types:
entity_types
aliquot    10171
Name: count, dtype: int64


In [8]:
# =============================================================================
# Validate file identity across GDC exports
# =============================================================================

metadata_files = pd.DataFrame(
    {
        "file_id": record["file_id"],
        "file_name": record["file_name"],
        "md5": record["md5sum"],
        "size": record["file_size"],
    }
    for record in metadata
)

manifest_ids = set(manifest["id"])
sample_sheet_ids = set(sample_sheet["File ID"])
metadata_ids = set(metadata_files["file_id"])

file_identity_checks = {
    "manifest_ids_unique": manifest["id"].is_unique,
    "sample_sheet_ids_unique": sample_sheet["File ID"].is_unique,
    "metadata_ids_unique": metadata_files["file_id"].is_unique,
    "manifest_equals_sample_sheet": manifest_ids == sample_sheet_ids,
    "manifest_equals_metadata": manifest_ids == metadata_ids,
}

for check_name, check_passed in file_identity_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(file_identity_checks.values()):
    raise ValueError(
        "GDC file identity is inconsistent across acquisition exports."
    )

manifest_ids_unique: True
sample_sheet_ids_unique: True
metadata_ids_unique: True
manifest_equals_sample_sheet: True
manifest_equals_metadata: True


In [9]:
# =============================================================================
# Validate manifest metadata against GDC metadata
# =============================================================================

manifest_metadata = (
    manifest[
        [
            "id",
            "filename",
            "md5",
            "size",
        ]
    ]
    .rename(
        columns={
            "id": "file_id",
            "filename": "manifest_file_name",
            "md5": "manifest_md5",
            "size": "manifest_size",
        }
    )
    .merge(
        metadata_files.rename(
            columns={
                "file_name": "metadata_file_name",
                "md5": "metadata_md5",
                "size": "metadata_size",
            }
        ),
        on="file_id",
        how="inner",
        validate="one_to_one",
    )
)

manifest_metadata_checks = {
    "row_count_matches_manifest": (
        len(manifest_metadata) == len(manifest)
    ),
    "file_names_match": (
        manifest_metadata["manifest_file_name"]
        .eq(manifest_metadata["metadata_file_name"])
        .all()
    ),
    "md5_values_match": (
        manifest_metadata["manifest_md5"]
        .eq(manifest_metadata["metadata_md5"])
        .all()
    ),
    "file_sizes_match": (
        manifest_metadata["manifest_size"]
        .eq(manifest_metadata["metadata_size"])
        .all()
    ),
}

for check_name, check_passed in manifest_metadata_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(manifest_metadata_checks.values()):
    raise ValueError(
        "Manifest and metadata disagree on GDC file identity."
    )

row_count_matches_manifest: True
file_names_match: True
md5_values_match: True
file_sizes_match: True


In [10]:
# =============================================================================
# Characterize GDC sample-sheet biological scope
# =============================================================================

print(f"Files:    {len(sample_sheet):,}")
print(f"Cases:    {sample_sheet['Case ID'].nunique():,}")
print(f"Samples:  {sample_sheet['Sample ID'].nunique():,}")
print(f"Projects: {sample_sheet['Project ID'].nunique():,}")

for column in [
    "Tissue Type",
    "Tumor Descriptor",
    "Specimen Type",
    "Preservation Method",
]:
    print(f"\n{column}:")
    print(
        sample_sheet[column]
        .value_counts(dropna=False)
    )

Files:    10,171
Cases:    9,808
Samples:  9,861
Projects: 33

Tissue Type:
Tissue Type
Tumor, Normal    5088
Normal, Tumor    5083
Name: count, dtype: int64

Tumor Descriptor:
Tumor Descriptor
Primary, Not Applicable    5086
Not Applicable, Primary    5083
Primary, Not Reported         2
Name: count, dtype: int64

Specimen Type:
Specimen Type
Solid Tissue, Peripheral Blood NOS    2995
Peripheral Blood NOS, Solid Tissue    2927
Peripheral Blood NOS, Unknown         1397
Unknown, Peripheral Blood NOS         1388
Solid Tissue, Solid Tissue             718
Unknown, Solid Tissue                  377
Solid Tissue, Unknown                  364
Solid Tissue, Buccal Cells               2
Buccal Cells, Unknown                    1
Unknown, Buccal Cells                    1
Buccal Cells, Peripheral Blood NOS       1
Name: count, dtype: int64

Preservation Method:
Preservation Method
Unknown, Unknown    6411
Unknown, OCT        1734
OCT, Unknown        1733
OCT, OCT             241
FFPE, Unknown

In [11]:
# =============================================================================
# Characterize associated TCGA aliquot sample-type codes
# =============================================================================

associated_entities = pd.DataFrame(
    {
        "file_id": record["file_id"],
        "case_id": entity["case_id"],
        "entity_id": entity["entity_id"],
        "entity_submitter_id": entity["entity_submitter_id"],
        "entity_type": entity["entity_type"],
    }
    for record in metadata
    for entity in record.get("associated_entities", [])
)

associated_entities["tcga_sample_type_code"] = (
    associated_entities["entity_submitter_id"]
    .astype("string")
    .str.extract(
        r"^TCGA-[^-]+-[^-]+-(\d{2})",
        expand=False,
    )
)

print(f"Associated aliquots: {len(associated_entities):,}")
print(
    "Unparsed sample-type codes: "
    f"{associated_entities['tcga_sample_type_code'].isna().sum():,}"
)

print("\nTCGA sample-type codes:")
print(
    associated_entities["tcga_sample_type_code"]
    .value_counts(dropna=False)
)

sample_type_combinations = (
    associated_entities
    .groupby("file_id")["tcga_sample_type_code"]
    .agg(
        lambda values: "|".join(
            sorted(values.dropna().unique())
        )
    )
)

print("\nSample-type-code combinations per file:")
print(sample_type_combinations.value_counts())

Associated aliquots: 20,342
Unparsed sample-type codes: 0

TCGA sample-type codes:
tcga_sample_type_code
01    10018
10     8555
11     1611
03      153
12        5
Name: count, dtype: int64[pyarrow]

Sample-type-code combinations per file:
tcga_sample_type_code
01|10    8555
01|11    1459
03|11     152
01|12       4
03|12       1
Name: count, dtype: int64[pyarrow]


In [12]:
# =============================================================================
# Validate primary-tumor aliquot composition
# =============================================================================

PRIMARY_TUMOR_CODES = {"01", "03"}
NORMAL_CODES = {"10", "11", "12"}

file_sample_types = (
    associated_entities
    .groupby("file_id")["tcga_sample_type_code"]
    .agg(
        lambda values: tuple(
            sorted(values.dropna().unique())
        )
    )
    .rename("sample_type_codes")
    .reset_index()
)

file_sample_types["has_primary_tumor"] = (
    file_sample_types["sample_type_codes"]
    .map(
        lambda codes: bool(
            PRIMARY_TUMOR_CODES.intersection(codes)
        )
    )
)

file_sample_types["has_normal"] = (
    file_sample_types["sample_type_codes"]
    .map(
        lambda codes: bool(
            NORMAL_CODES.intersection(codes)
        )
    )
)

primary_scope_checks = {
    "all_files_have_primary_tumor": (
        file_sample_types["has_primary_tumor"].all()
    ),
    "all_files_have_normal": (
        file_sample_types["has_normal"].all()
    ),
    "all_files_have_two_sample_types": (
        file_sample_types["sample_type_codes"]
        .map(len)
        .eq(2)
        .all()
    ),
}

for check_name, check_passed in primary_scope_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(primary_scope_checks.values()):
    raise ValueError(
        "Mutation files do not uniformly represent "
        "primary-tumor/normal analyses."
    )

all_files_have_primary_tumor: True
all_files_have_normal: True
all_files_have_two_sample_types: True


In [13]:
# =============================================================================
# Characterize mutation-file multiplicity per TCGA case
# =============================================================================

file_case_mapping = (
    associated_entities[
        [
            "file_id",
            "case_id",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

files_per_case = (
    file_case_mapping
    .groupby("case_id")
    .size()
    .rename("n_files")
)

print(f"Cases represented: {len(files_per_case):,}")
print(f"Files represented: {len(file_case_mapping):,}")

print("\nFiles per case:")
print(
    files_per_case
    .value_counts()
    .sort_index()
)

print(
    "\nCases with more than one mutation file: "
    f"{files_per_case.gt(1).sum():,}"
)

print(
    "Maximum files for a single case: "
    f"{files_per_case.max():,}"
)

Cases represented: 9,808
Files represented: 10,171

Files per case:
n_files
1    9498
2     279
3      19
4       5
5       4
6       3
Name: count, dtype: int64

Cases with more than one mutation file: 310
Maximum files for a single case: 6


In [14]:
# =============================================================================
# Characterize tumor-normal pairing multiplicity
# =============================================================================

primary_entities = (
    associated_entities.loc[
        associated_entities["tcga_sample_type_code"].isin(
            PRIMARY_TUMOR_CODES
        )
    ]
    .rename(
        columns={
            "entity_id": "tumor_aliquot_id",
            "entity_submitter_id": "tumor_aliquot_barcode",
        }
    )
)

normal_entities = (
    associated_entities.loc[
        associated_entities["tcga_sample_type_code"].isin(
            NORMAL_CODES
        )
    ]
    .rename(
        columns={
            "entity_id": "normal_aliquot_id",
            "entity_submitter_id": "normal_aliquot_barcode",
        }
    )
)

tumor_normal_pairs = (
    primary_entities[
        [
            "file_id",
            "case_id",
            "tumor_aliquot_id",
            "tumor_aliquot_barcode",
        ]
    ]
    .merge(
        normal_entities[
            [
                "file_id",
                "normal_aliquot_id",
                "normal_aliquot_barcode",
            ]
        ],
        on="file_id",
        how="inner",
        validate="one_to_one",
    )
)

case_pair_multiplicity = (
    tumor_normal_pairs
    .groupby("case_id")
    .agg(
        n_files=("file_id", "nunique"),
        n_tumor_aliquots=("tumor_aliquot_id", "nunique"),
        n_normal_aliquots=("normal_aliquot_id", "nunique"),
    )
)

multi_case_pair_multiplicity = (
    case_pair_multiplicity.loc[
        case_pair_multiplicity["n_files"].gt(1)
    ]
)

print("Multiplicity patterns among multi-file cases:")
print(
    multi_case_pair_multiplicity[
        [
            "n_files",
            "n_tumor_aliquots",
            "n_normal_aliquots",
        ]
    ]
    .value_counts()
    .sort_index()
)

Multiplicity patterns among multi-file cases:
n_files  n_tumor_aliquots  n_normal_aliquots
2        2                 1                    279
3        3                 1                     19
4        4                 1                      5
5        5                 1                      4
6        6                 1                      3
Name: count, dtype: int64


In [15]:
# =============================================================================
# Characterize hierarchical tumor biospecimen multiplicity
# =============================================================================

tumor_normal_pairs["tumor_sample_barcode"] = (
    tumor_normal_pairs["tumor_aliquot_barcode"]
    .astype("string")
    .str.extract(
        r"^(TCGA-[^-]+-[^-]+-\d{2}[A-Z])",
        expand=False,
    )
)

tumor_normal_pairs["tumor_portion_barcode"] = (
    tumor_normal_pairs["tumor_aliquot_barcode"]
    .astype("string")
    .str.extract(
        r"^(TCGA-[^-]+-[^-]+-\d{2}[A-Z]-[^-]+)",
        expand=False,
    )
)

tumor_biospecimen_multiplicity = (
    tumor_normal_pairs
    .groupby("case_id")
    .agg(
        n_files=("file_id", "nunique"),
        n_tumor_samples=("tumor_sample_barcode", "nunique"),
        n_tumor_portions=("tumor_portion_barcode", "nunique"),
        n_tumor_aliquots=("tumor_aliquot_id", "nunique"),
    )
)

print("Tumor biospecimen patterns among multi-file cases:")
print(
    tumor_biospecimen_multiplicity.loc[
        tumor_biospecimen_multiplicity["n_files"].gt(1)
    ]
    .value_counts()
    .sort_index()
)

Tumor biospecimen patterns among multi-file cases:
n_files  n_tumor_samples  n_tumor_portions  n_tumor_aliquots
2        1                1                 2                    57
                          2                 2                   198
         2                2                 2                    24
3        1                1                 3                     1
                          3                 3                     1
         2                2                 3                    16
                          3                 3                     1
4        2                3                 4                     5
5        2                4                 5                     4
6        2                4                 6                     1
                          5                 6                     2
Name: count, dtype: int64


In [16]:
# =============================================================================
# Load frozen TCGA multi-omic sample mapping
# =============================================================================

FROZEN_MULTIIOMIC_SAMPLE_MAPPING_PATH = (
    Paths.metadata
    / "tcga_primary_tumor_multiomic_final_case_level_sample_mapping.csv"
)

frozen_multiomic_mapping = pd.read_csv(
    FROZEN_MULTIIOMIC_SAMPLE_MAPPING_PATH,
)

print(f"Frozen multi-omic cases: {len(frozen_multiomic_mapping):,}")

print("\nColumns:")
print(frozen_multiomic_mapping.columns.tolist())

Frozen multi-omic cases: 9,965

Columns:
['final_sample_column_index', 'case_submitter_id', 'sample_submitter_id', 'project_id', 'rna_case_uuid', 'rna_aliquot_uuid', 'rna_aliquot_submitter_id', 'rna_file_id', 'rna_file_name', 'rna_candidate_matrix_column_index', 'rna_final_matrix_column_index', 'rna_qc_eligible_for_downstream_selection', 'gene_assigned_fraction_of_accounted', 'methylation_case_uuid', 'methylation_aliquot_uuid', 'methylation_aliquot_submitter_id', 'methylation_file_id', 'methylation_file_name', 'methylation_platform', 'hm27_matrix_column_index', 'hm450_matrix_column_index', 'shared_matrix_column_index', 'missing_beta_fraction', 'methylation_qc_eligible_for_downstream_selection', 'pair_eligible_after_methylation_qc', 'case_level_row_decision', 'rna_local_path', 'methylation_local_path', 'rna_payload_exists', 'methylation_payload_exists']


In [17]:
# =============================================================================
# Prepare frozen TCGA case-sample reference
# =============================================================================

frozen_case_sample_reference = (
    frozen_multiomic_mapping[
        [
            "case_submitter_id",
            "sample_submitter_id",
            "project_id",
        ]
    ]
    .rename(
        columns={
            "case_submitter_id": "tcga_case_barcode",
            "sample_submitter_id": "frozen_tumor_sample_barcode",
        }
    )
    .copy()
)

print(
    "Frozen case-sample references: "
    f"{len(frozen_case_sample_reference):,}"
)

print("\nFrozen tumor sample-type codes:")
print(
    frozen_case_sample_reference[
        "frozen_tumor_sample_barcode"
    ]
    .str.extract(
        r"^TCGA-[^-]+-[^-]+-(\d{2})",
        expand=False,
    )
    .value_counts(dropna=False)
)

Frozen case-sample references: 9,965

Frozen tumor sample-type codes:
frozen_tumor_sample_barcode
01    9831
03     134
Name: count, dtype: int64


In [18]:
# =============================================================================
# Compare mutation coverage with the frozen TCGA cohort
# =============================================================================

mutation_case_reference = (
    tumor_normal_pairs[
        [
            "case_id",
            "tumor_aliquot_barcode",
        ]
    ]
    .assign(
        tcga_case_barcode=lambda frame: (
            frame["tumor_aliquot_barcode"]
            .astype("string")
            .str.extract(
                r"^(TCGA-[^-]+-[^-]+)",
                expand=False,
            )
        )
    )
    [
        [
            "case_id",
            "tcga_case_barcode",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

frozen_cases = set(
    frozen_case_sample_reference["tcga_case_barcode"]
)

mutation_cases = set(
    mutation_case_reference["tcga_case_barcode"]
)

shared_cases = frozen_cases & mutation_cases
frozen_only_cases = frozen_cases - mutation_cases
mutation_only_cases = mutation_cases - frozen_cases

print(f"Frozen multi-omic cases: {len(frozen_cases):,}")
print(f"Mutation-resource cases: {len(mutation_cases):,}")
print(f"Shared cases:            {len(shared_cases):,}")
print(f"Frozen only:             {len(frozen_only_cases):,}")
print(f"Mutation only:           {len(mutation_only_cases):,}")

print(
    "\nFrozen-cohort mutation coverage: "
    f"{len(shared_cases) / len(frozen_cases):.3%}"
)

Frozen multi-omic cases: 9,965
Mutation-resource cases: 9,808
Shared cases:            9,230
Frozen only:             735
Mutation only:           578

Frozen-cohort mutation coverage: 92.624%


In [19]:
# =============================================================================
# Match mutation tumor samples to the frozen TCGA cohort
# =============================================================================

mutation_file_sample_mapping = (
    tumor_normal_pairs[
        [
            "file_id",
            "case_id",
            "tumor_sample_barcode",
            "tumor_aliquot_id",
            "tumor_aliquot_barcode",
            "normal_aliquot_id",
            "normal_aliquot_barcode",
        ]
    ]
    .assign(
        tcga_case_barcode=lambda frame: (
            frame["tumor_aliquot_barcode"]
            .astype("string")
            .str.extract(
                r"^(TCGA-[^-]+-[^-]+)",
                expand=False,
            )
        )
    )
    .merge(
        frozen_case_sample_reference,
        on="tcga_case_barcode",
        how="left",
        validate="many_to_one",
    )
)

mutation_file_sample_mapping["exact_frozen_sample_match"] = (
    mutation_file_sample_mapping["tumor_sample_barcode"]
    .eq(
        mutation_file_sample_mapping[
            "frozen_tumor_sample_barcode"
        ]
    )
)

shared_case_files = (
    mutation_file_sample_mapping[
        "frozen_tumor_sample_barcode"
    ]
    .notna()
)

exact_match_files = (
    mutation_file_sample_mapping[
        "exact_frozen_sample_match"
    ]
)

exact_match_cases = (
    mutation_file_sample_mapping.loc[
        exact_match_files,
        "tcga_case_barcode",
    ]
    .nunique()
)

shared_case_without_exact_sample = (
    mutation_file_sample_mapping.loc[
        shared_case_files,
        "tcga_case_barcode",
    ]
    .drop_duplicates()
    .loc[
        lambda series: ~series.isin(
            mutation_file_sample_mapping.loc[
                exact_match_files,
                "tcga_case_barcode",
            ]
        )
    ]
)

print(
    "Mutation files from frozen cases: "
    f"{shared_case_files.sum():,}"
)
print(
    "Files matching the exact frozen tumor sample: "
    f"{exact_match_files.sum():,}"
)
print(
    "Frozen cases with an exact tumor-sample match: "
    f"{exact_match_cases:,}"
)
print(
    "Shared cases without an exact tumor-sample match: "
    f"{shared_case_without_exact_sample.nunique():,}"
)

Mutation files from frozen cases: 9,507
Files matching the exact frozen tumor sample: 9,426
Frozen cases with an exact tumor-sample match: 9,192
Shared cases without an exact tumor-sample match: 38


In [20]:
# =============================================================================
# Characterize multiplicity after exact frozen-sample matching
# =============================================================================

exact_sample_files = (
    mutation_file_sample_mapping.loc[
        mutation_file_sample_mapping["exact_frozen_sample_match"]
    ]
    .copy()
)

exact_files_per_case = (
    exact_sample_files
    .groupby("tcga_case_barcode")
    .agg(
        n_files=("file_id", "nunique"),
        n_tumor_aliquots=("tumor_aliquot_id", "nunique"),
        n_normal_aliquots=("normal_aliquot_id", "nunique"),
    )
)

print("Exact-match files per frozen case:")
print(
    exact_files_per_case["n_files"]
    .value_counts()
    .sort_index()
)

print(
    "\nCases retaining more than one exact-match MAF: "
    f"{exact_files_per_case['n_files'].gt(1).sum():,}"
)

print("\nMultiplicity patterns among those cases:")
print(
    exact_files_per_case.loc[
        exact_files_per_case["n_files"].gt(1)
    ]
    .value_counts()
    .sort_index()
)

Exact-match files per frozen case:
n_files
1    8977
2     204
3       5
4       4
5       2
Name: count, dtype: int64

Cases retaining more than one exact-match MAF: 215

Multiplicity patterns among those cases:
n_files  n_tumor_aliquots  n_normal_aliquots
2        2                 1                    204
3        3                 1                      5
4        4                 1                      4
5        5                 1                      2
Name: count, dtype: int64


In [22]:
# =============================================================================
# Parse TCGA tumor aliquot hierarchy
# =============================================================================

tumor_barcode_fields = (
    exact_sample_files["tumor_aliquot_barcode"]
    .astype("string")
    .str.extract(
        (
            r"^(?P<participant_barcode>TCGA-[^-]+-[^-]+)-"
            r"(?P<sample_type_code>\d{2})"
            r"(?P<vial>[A-Z])-"
            r"(?P<portion>\d{2})"
            r"(?P<analyte>[A-Z])-"
            r"(?P<plate>[^-]+)-"
            r"(?P<center>[^-]+)$"
        )
    )
)

if tumor_barcode_fields.isna().any(axis=None):
    raise ValueError(
        "At least one tumor aliquot barcode could not be parsed."
    )

exact_sample_files["tumor_portion_barcode"] = (
    tumor_barcode_fields["participant_barcode"]
    + "-"
    + tumor_barcode_fields["sample_type_code"]
    + tumor_barcode_fields["vial"]
    + "-"
    + tumor_barcode_fields["portion"]
)

exact_sample_files["tumor_analyte"] = (
    tumor_barcode_fields["analyte"]
)

true_portion_multiplicity = (
    exact_sample_files
    .groupby("tcga_case_barcode")
    .agg(
        n_files=("file_id", "nunique"),
        n_tumor_portions=("tumor_portion_barcode", "nunique"),
        n_tumor_aliquots=("tumor_aliquot_id", "nunique"),
    )
)

print("Residual biospecimen patterns:")
print(
    true_portion_multiplicity.loc[
        true_portion_multiplicity["n_files"].gt(1)
    ]
    .value_counts()
    .sort_index()
)

print("\nTumor analytes:")
print(
    exact_sample_files["tumor_analyte"]
    .value_counts(dropna=False)
)

Residual biospecimen patterns:
n_files  n_tumor_portions  n_tumor_aliquots
2        1                 2                   204
3        1                 3                     3
         2                 3                     2
4        3                 4                     4
5        3                 5                     2
Name: count, dtype: int64

Tumor analytes:
tumor_analyte
D    8432
W     994
Name: count, dtype: int64[pyarrow]


In [23]:
# =============================================================================
# Compare mutation portions with frozen multi-omic assay portions
# =============================================================================

frozen_assay_reference = (
    frozen_multiomic_mapping[
        [
            "case_submitter_id",
            "rna_aliquot_submitter_id",
            "methylation_aliquot_submitter_id",
        ]
    ]
    .rename(
        columns={
            "case_submitter_id": "tcga_case_barcode",
        }
    )
    .copy()
)


def extract_tcga_portion_barcode(barcodes):
    return (
        barcodes
        .astype("string")
        .str.extract(
            r"^(TCGA-[^-]+-[^-]+-\d{2}[A-Z]-\d{2})",
            expand=False,
        )
    )


frozen_assay_reference["rna_portion_barcode"] = (
    extract_tcga_portion_barcode(
        frozen_assay_reference["rna_aliquot_submitter_id"]
    )
)

frozen_assay_reference["methylation_portion_barcode"] = (
    extract_tcga_portion_barcode(
        frozen_assay_reference["methylation_aliquot_submitter_id"]
    )
)

exact_sample_files = (
    exact_sample_files
    .merge(
        frozen_assay_reference[
            [
                "tcga_case_barcode",
                "rna_portion_barcode",
                "methylation_portion_barcode",
            ]
        ],
        on="tcga_case_barcode",
        how="left",
        validate="many_to_one",
    )
)

exact_sample_files["matches_rna_portion"] = (
    exact_sample_files["tumor_portion_barcode"]
    .eq(exact_sample_files["rna_portion_barcode"])
)

exact_sample_files["matches_methylation_portion"] = (
    exact_sample_files["tumor_portion_barcode"]
    .eq(exact_sample_files["methylation_portion_barcode"])
)

print(
    "Mutation files matching frozen methylation portion: "
    f"{exact_sample_files['matches_methylation_portion'].sum():,}"
)

print(
    "Mutation files matching frozen RNA portion: "
    f"{exact_sample_files['matches_rna_portion'].sum():,}"
)

print("\nAmong cases with multiple exact-sample MAFs:")
print(
    exact_sample_files.loc[
        exact_sample_files["tcga_case_barcode"].isin(
            exact_files_per_case.index[
                exact_files_per_case["n_files"].gt(1)
            ]
        )
    ]
    .groupby("tcga_case_barcode")
    .agg(
        n_files=("file_id", "nunique"),
        n_methylation_portion_matches=(
            "matches_methylation_portion",
            "sum",
        ),
        n_rna_portion_matches=(
            "matches_rna_portion",
            "sum",
        ),
    )
    .value_counts()
    .sort_index()
)

Mutation files matching frozen methylation portion: 9,407
Mutation files matching frozen RNA portion: 9,407

Among cases with multiple exact-sample MAFs:
n_files  n_methylation_portion_matches  n_rna_portion_matches
2        0                              0                          2
         2                              2                        202
3        2                              2                          2
         3                              3                          3
4        2                              2                          4
5        3                              3                          2
Name: count, dtype: int64


In [24]:
# =============================================================================
# Characterize analyte composition in residual exact-sample multiplicity
# =============================================================================

multi_exact_cases = (
    exact_files_per_case.index[
        exact_files_per_case["n_files"].gt(1)
    ]
)

multi_exact_files = (
    exact_sample_files.loc[
        exact_sample_files["tcga_case_barcode"].isin(
            multi_exact_cases
        )
    ]
    .copy()
)

analyte_patterns = (
    multi_exact_files
    .groupby("tcga_case_barcode")
    .agg(
        n_files=("file_id", "nunique"),
        analytes=(
            "tumor_analyte",
            lambda values: "|".join(
                sorted(values.dropna().unique())
            ),
        ),
        n_analytes=("tumor_analyte", "nunique"),
        n_portions=("tumor_portion_barcode", "nunique"),
        n_frozen_portion_matches=(
            "matches_methylation_portion",
            "sum",
        ),
    )
)

print("Analyte patterns among multi-file exact-sample cases:")
print(
    analyte_patterns
    .value_counts()
    .sort_index()
)

Analyte patterns among multi-file exact-sample cases:
n_files  analytes  n_analytes  n_portions  n_frozen_portion_matches
2        D         1           1           2                            17
         D|W       2           1           0                             2
                                           2                           156
         W         1           1           2                            29
3        D         1           2           2                             1
         D|W       2           1           3                             2
                               2           2                             1
         W         1           1           3                             1
4        D         1           3           2                             4
5        D         1           3           3                             1
         D|W       2           3           3                             1
Name: count, dtype: int64


In [26]:
# =============================================================================
# Compare mutation aliquots with the frozen methylation aliquot
# =============================================================================

frozen_methylation_aliquots = (
    frozen_multiomic_mapping[
        [
            "case_submitter_id",
            "methylation_aliquot_submitter_id",
        ]
    ]
    .rename(
        columns={
            "case_submitter_id": "tcga_case_barcode",
            "methylation_aliquot_submitter_id": (
                "frozen_methylation_aliquot_barcode"
            ),
        }
    )
    .copy()
)

exact_sample_files = (
    exact_sample_files
    .merge(
        frozen_methylation_aliquots,
        on="tcga_case_barcode",
        how="left",
        validate="many_to_one",
    )
)

exact_sample_files["matches_methylation_aliquot"] = (
    exact_sample_files["tumor_aliquot_barcode"]
    .eq(
        exact_sample_files[
            "frozen_methylation_aliquot_barcode"
        ]
    )
)

n_matching_files = (
    exact_sample_files["matches_methylation_aliquot"]
    .sum()
)

n_matching_cases = (
    exact_sample_files.loc[
        exact_sample_files["matches_methylation_aliquot"],
        "tcga_case_barcode",
    ]
    .nunique()
)

print(
    "Mutation files matching the exact frozen methylation aliquot: "
    f"{n_matching_files:,}"
)

print(
    "Frozen cases with an exact methylation-aliquot match: "
    f"{n_matching_cases:,}"
)

print("\nAmong multi-file exact-sample cases:")
print(
    exact_sample_files.loc[
        exact_sample_files["tcga_case_barcode"].isin(
            multi_exact_cases
        )
    ]
    .groupby("tcga_case_barcode")
    .agg(
        n_files=("file_id", "nunique"),
        n_exact_methylation_aliquot_matches=(
            "matches_methylation_aliquot",
            "sum",
        ),
    )
    .value_counts()
    .sort_index()
)

Mutation files matching the exact frozen methylation aliquot: 0
Frozen cases with an exact methylation-aliquot match: 0

Among multi-file exact-sample cases:
n_files  n_exact_methylation_aliquot_matches
2        0                                      204
3        0                                        5
4        0                                        4
5        0                                        2
Name: count, dtype: int64


In [27]:
# =============================================================================
# Classify frozen cases by mutation-resource eligibility
# =============================================================================

exact_file_counts = (
    exact_sample_files
    .groupby("tcga_case_barcode")["file_id"]
    .nunique()
)

case_mutation_status = (
    frozen_case_sample_reference[
        [
            "tcga_case_barcode",
            "frozen_tumor_sample_barcode",
            "project_id",
        ]
    ]
    .copy()
)

case_mutation_status["has_mutation_case"] = (
    case_mutation_status["tcga_case_barcode"]
    .isin(mutation_cases)
)

case_mutation_status["n_exact_sample_mafs"] = (
    case_mutation_status["tcga_case_barcode"]
    .map(exact_file_counts)
    .fillna(0)
    .astype("int64")
)

case_mutation_status["mutation_resource_status"] = (
    "no_mutation_resource"
)

case_mutation_status.loc[
    case_mutation_status["has_mutation_case"]
    & case_mutation_status["n_exact_sample_mafs"].eq(0),
    "mutation_resource_status",
] = "no_exact_frozen_sample_match"

case_mutation_status.loc[
    case_mutation_status["n_exact_sample_mafs"].eq(1),
    "mutation_resource_status",
] = "eligible_single_exact_sample_maf"

case_mutation_status.loc[
    case_mutation_status["n_exact_sample_mafs"].gt(1),
    "mutation_resource_status",
] = "multiple_exact_sample_mafs"

print("Frozen-case mutation-resource status:")
print(
    case_mutation_status["mutation_resource_status"]
    .value_counts()
)

print(
    "\nStrict mutation-resource coverage: "
    f"{case_mutation_status['n_exact_sample_mafs'].eq(1).mean():.3%}"
)

Frozen-case mutation-resource status:
mutation_resource_status
eligible_single_exact_sample_maf    8977
no_mutation_resource                 735
multiple_exact_sample_mafs           215
no_exact_frozen_sample_match          38
Name: count, dtype: int64

Strict mutation-resource coverage: 90.085%


In [28]:
# =============================================================================
# Characterize mutation-resource coverage by TCGA project
# =============================================================================

project_mutation_coverage = (
    case_mutation_status
    .groupby("project_id")
    .agg(
        n_frozen_cases=("tcga_case_barcode", "size"),
        n_eligible=(
            "mutation_resource_status",
            lambda values: (
                values == "eligible_single_exact_sample_maf"
            ).sum(),
        ),
        n_no_resource=(
            "mutation_resource_status",
            lambda values: (
                values == "no_mutation_resource"
            ).sum(),
        ),
        n_no_exact_sample_match=(
            "mutation_resource_status",
            lambda values: (
                values == "no_exact_frozen_sample_match"
            ).sum(),
        ),
        n_multiple_exact_mafs=(
            "mutation_resource_status",
            lambda values: (
                values == "multiple_exact_sample_mafs"
            ).sum(),
        ),
    )
    .reset_index()
)

project_mutation_coverage["eligible_fraction"] = (
    project_mutation_coverage["n_eligible"]
    / project_mutation_coverage["n_frozen_cases"]
)

project_mutation_coverage = (
    project_mutation_coverage
    .sort_values(
        "eligible_fraction",
        ascending=True,
        kind="stable",
    )
    .reset_index(drop=True)
)

print(
    project_mutation_coverage.to_string(
        index=False,
        formatters={
            "eligible_fraction": "{:.3%}".format,
        },
    )
)

project_id  n_frozen_cases  n_eligible  n_no_resource  n_no_exact_sample_match  n_multiple_exact_mafs eligible_fraction
 TCGA-LAML             134          65             30                       35                      4           48.507%
   TCGA-OV             422         249            133                        0                     40           59.005%
  TCGA-GBM             230         144             24                        2                     60           62.609%
 TCGA-KIRC             487         314            158                        0                     15           64.476%
 TCGA-DLBC              48          36             12                        0                      0           75.000%
 TCGA-READ             164         143             16                        0                      5           87.195%
 TCGA-LUSC             500         437             14                        0                     49           87.400%
 TCGA-BRCA            1089         955  

In [29]:
# =============================================================================
# Define primary mutation-resource eligibility
# =============================================================================

case_mutation_status["mutation_resource_status"] = (
    "no_mutation_resource"
)

case_mutation_status.loc[
    case_mutation_status["has_mutation_case"]
    & case_mutation_status["n_exact_sample_mafs"].eq(0),
    "mutation_resource_status",
] = "no_exact_frozen_sample_match"

case_mutation_status.loc[
    case_mutation_status["n_exact_sample_mafs"].eq(1),
    "mutation_resource_status",
] = "eligible_single_exact_sample_maf"

case_mutation_status.loc[
    case_mutation_status["n_exact_sample_mafs"].gt(1),
    "mutation_resource_status",
] = "eligible_multiple_exact_sample_mafs"

case_mutation_status["mutation_resource_eligible"] = (
    case_mutation_status["n_exact_sample_mafs"].ge(1)
)

print("Frozen-case mutation-resource status:")
print(
    case_mutation_status["mutation_resource_status"]
    .value_counts()
)

print(
    "\nPrimary mutation-resource eligible cases: "
    f"{case_mutation_status['mutation_resource_eligible'].sum():,}"
)

print(
    "Primary mutation-resource coverage: "
    f"{case_mutation_status['mutation_resource_eligible'].mean():.3%}"
)

Frozen-case mutation-resource status:
mutation_resource_status
eligible_single_exact_sample_maf       8977
no_mutation_resource                    735
eligible_multiple_exact_sample_mafs     215
no_exact_frozen_sample_match             38
Name: count, dtype: int64

Primary mutation-resource eligible cases: 9,192
Primary mutation-resource coverage: 92.243%


In [30]:
# =============================================================================
# Characterize primary mutation-resource coverage by TCGA project
# =============================================================================

project_primary_coverage = (
    case_mutation_status
    .groupby("project_id")
    .agg(
        n_frozen_cases=("tcga_case_barcode", "size"),
        n_eligible=("mutation_resource_eligible", "sum"),
        n_no_resource=(
            "mutation_resource_status",
            lambda values: (
                values == "no_mutation_resource"
            ).sum(),
        ),
        n_no_exact_sample_match=(
            "mutation_resource_status",
            lambda values: (
                values == "no_exact_frozen_sample_match"
            ).sum(),
        ),
        n_multiple_exact_mafs=(
            "mutation_resource_status",
            lambda values: (
                values == "eligible_multiple_exact_sample_mafs"
            ).sum(),
        ),
    )
    .reset_index()
)

project_primary_coverage["eligible_fraction"] = (
    project_primary_coverage["n_eligible"]
    / project_primary_coverage["n_frozen_cases"]
)

project_primary_coverage = (
    project_primary_coverage
    .sort_values(
        "eligible_fraction",
        ascending=True,
        kind="stable",
    )
    .reset_index(drop=True)
)

print(
    project_primary_coverage.to_string(
        index=False,
        formatters={
            "eligible_fraction": "{:.3%}".format,
        },
    )
)

project_id  n_frozen_cases  n_eligible  n_no_resource  n_no_exact_sample_match  n_multiple_exact_mafs eligible_fraction
 TCGA-LAML             134          69             30                       35                      4           51.493%
 TCGA-KIRC             487         329            158                        0                     15           67.556%
   TCGA-OV             422         289            133                        0                     40           68.483%
 TCGA-DLBC              48          36             12                        0                      0           75.000%
 TCGA-BRCA            1089         960            129                        0                      5           88.154%
  TCGA-GBM             230         204             24                        2                     60           88.696%
 TCGA-SARC             259         232             27                        0                      0           89.575%
 TCGA-READ             164         148  

In [31]:
# =============================================================================
# Characterize cases without an exact frozen tumor-sample match
# =============================================================================

mismatch_cases = (
    case_mutation_status.loc[
        case_mutation_status["mutation_resource_status"]
        .eq("no_exact_frozen_sample_match"),
        [
            "tcga_case_barcode",
            "frozen_tumor_sample_barcode",
            "project_id",
        ],
    ]
    .copy()
)

available_mutation_samples = (
    mutation_file_sample_mapping.loc[
        mutation_file_sample_mapping["tcga_case_barcode"]
        .isin(mismatch_cases["tcga_case_barcode"])
    ]
    .groupby("tcga_case_barcode")
    .agg(
        n_mafs=("file_id", "nunique"),
        n_mutation_samples=("tumor_sample_barcode", "nunique"),
        mutation_tumor_samples=(
            "tumor_sample_barcode",
            lambda values: "|".join(
                sorted(values.dropna().unique())
            ),
        ),
    )
    .reset_index()
)

mismatch_case_details = (
    mismatch_cases
    .merge(
        available_mutation_samples,
        on="tcga_case_barcode",
        how="left",
        validate="one_to_one",
    )
)

mismatch_case_details["frozen_sample_type_code"] = (
    mismatch_case_details["frozen_tumor_sample_barcode"]
    .str.extract(
        r"^TCGA-[^-]+-[^-]+-(\d{2})",
        expand=False,
    )
)

mismatch_case_details["available_sample_type_codes"] = (
    mismatch_case_details["mutation_tumor_samples"]
    .str.findall(
        r"TCGA-[^-]+-[^-]+-(\d{2})[A-Z]"
    )
    .map(
        lambda values: "|".join(sorted(set(values)))
    )
)

print("Mismatch cases by project:")
print(
    mismatch_case_details["project_id"]
    .value_counts()
)

print("\nFrozen versus available sample-type codes:")
print(
    mismatch_case_details[
        [
            "frozen_sample_type_code",
            "available_sample_type_codes",
        ]
    ]
    .value_counts()
)

print("\nMismatch-case details:")
print(
    mismatch_case_details.to_string(index=False)
)

Mismatch cases by project:
project_id
TCGA-LAML    35
TCGA-GBM      2
TCGA-COAD     1
Name: count, dtype: int64

Frozen versus available sample-type codes:
frozen_sample_type_code  available_sample_type_codes
03                       03                             35
01                       01                              3
Name: count, dtype: int64

Mismatch-case details:
tcga_case_barcode frozen_tumor_sample_barcode project_id  n_mafs  n_mutation_samples            mutation_tumor_samples frozen_sample_type_code available_sample_type_codes
     TCGA-06-0210            TCGA-06-0210-01A   TCGA-GBM       1                   1                  TCGA-06-0210-01B                      01                          01
     TCGA-06-0211            TCGA-06-0211-01A   TCGA-GBM       1                   1                  TCGA-06-0211-01B                      01                          01
     TCGA-A6-2672            TCGA-A6-2672-01A  TCGA-COAD       2                   1                  TCGA-A6-

## Frozen mutation-resource matching policy

Compatibility with the frozen TCGA multi-omic cohort is defined at the exact
TCGA tumor-sample barcode level rather than at case or sample-type level alone.

A mutation file is therefore eligible for the primary Phase 4B handoff only
when its tumor sample matches the `sample_submitter_id` already frozen for that
case in the 9,965-case multi-omic cohort.

Multiple mutation files corresponding to the same exact frozen tumor sample
are retained rather than arbitrarily reduced to one file. Their multiplicity
is treated as biospecimen/technical provenance and will remain explicit in the
handoff.

Cases for which mutation data are available only from a different vial of the
same TCGA sample type are not rescued for the primary analysis. The vial is
part of the TCGA sample identifier, and equivalence with the frozen sample
cannot be assumed solely from shared case and sample-type codes.

Under this policy:

- 9,192 frozen cases have at least one exact-sample mutation MAF and are
  primary mutation-resource eligible;
- 8,977 have exactly one eligible MAF;
- 215 have multiple eligible MAFs from the exact frozen tumor sample;
- 38 have mutation data only from a different tumor vial and remain excluded
  from the primary handoff; and
- 735 frozen cases have no mutation resource in the selected GDC snapshot.

The 38 alternate-vial cases remain traceable for audit and optional secondary
sensitivity characterization but cannot redefine primary eligibility.

No mutation-content information is used to determine sample eligibility or
resolve file multiplicity.

In [32]:
# =============================================================================
# Construct primary mutation file handoff
# =============================================================================

primary_mutation_file_handoff = (
    exact_sample_files[
        [
            "file_id",
            "case_id",
            "tcga_case_barcode",
            "project_id",
            "frozen_tumor_sample_barcode",
            "tumor_sample_barcode",
            "tumor_aliquot_id",
            "tumor_aliquot_barcode",
            "normal_aliquot_id",
            "normal_aliquot_barcode",
            "tumor_portion_barcode",
            "tumor_analyte",
        ]
    ]
    .rename(
        columns={
            "case_id": "gdc_case_id",
        }
    )
    .merge(
        manifest[
            [
                "id",
                "filename",
                "md5",
                "size",
            ]
        ].rename(
            columns={
                "id": "file_id",
                "filename": "file_name",
                "md5": "file_md5",
                "size": "file_size",
            }
        ),
        on="file_id",
        how="left",
        validate="one_to_one",
    )
)

primary_mutation_file_handoff["n_mafs_for_frozen_sample"] = (
    primary_mutation_file_handoff["tcga_case_barcode"]
    .map(
        primary_mutation_file_handoff
        .groupby("tcga_case_barcode")["file_id"]
        .nunique()
    )
)

print(
    "Primary mutation MAFs: "
    f"{len(primary_mutation_file_handoff):,}"
)

print(
    "Primary mutation cases: "
    f"{primary_mutation_file_handoff['tcga_case_barcode'].nunique():,}"
)

print("\nMAFs per frozen tumor sample:")
print(
    primary_mutation_file_handoff["n_mafs_for_frozen_sample"]
    .value_counts()
    .sort_index()
)

Primary mutation MAFs: 9,426
Primary mutation cases: 9,192

MAFs per frozen tumor sample:
n_mafs_for_frozen_sample
1    8977
2     408
3      15
4      16
5      10
Name: count, dtype: int64


In [33]:
# =============================================================================
# Validate primary mutation file handoff
# =============================================================================

primary_handoff_checks = {
    "file_ids_are_unique": (
        primary_mutation_file_handoff["file_id"].is_unique
    ),
    "all_files_match_frozen_sample": (
        primary_mutation_file_handoff["tumor_sample_barcode"]
        .eq(
            primary_mutation_file_handoff[
                "frozen_tumor_sample_barcode"
            ]
        )
        .all()
    ),
    "all_files_have_manifest_identity": (
        primary_mutation_file_handoff[
            [
                "file_name",
                "file_md5",
                "file_size",
            ]
        ]
        .notna()
        .all(axis=None)
    ),
    "case_count_matches_primary_eligibility": (
        primary_mutation_file_handoff["tcga_case_barcode"]
        .nunique()
        == case_mutation_status[
            "mutation_resource_eligible"
        ].sum()
    ),
    "alternate_vial_cases_are_excluded": (
        not primary_mutation_file_handoff[
            "tcga_case_barcode"
        ]
        .isin(
            mismatch_case_details["tcga_case_barcode"]
        )
        .any()
    ),
}

for check_name, check_passed in primary_handoff_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(primary_handoff_checks.values()):
    raise ValueError(
        "Primary mutation file handoff validation failed."
    )

file_ids_are_unique: True
all_files_match_frozen_sample: True
all_files_have_manifest_identity: True
case_count_matches_primary_eligibility: True
alternate_vial_cases_are_excluded: True


In [34]:
# =============================================================================
# Construct primary mutation download manifest
# =============================================================================

primary_download_manifest = (
    manifest.loc[
        manifest["id"].isin(
            primary_mutation_file_handoff["file_id"]
        )
    ]
    .copy()
    .reset_index(drop=True)
)

primary_download_manifest_checks = {
    "row_count_matches_handoff": (
        len(primary_download_manifest)
        == len(primary_mutation_file_handoff)
    ),
    "file_ids_match_handoff": (
        set(primary_download_manifest["id"])
        == set(primary_mutation_file_handoff["file_id"])
    ),
    "file_ids_are_unique": (
        primary_download_manifest["id"].is_unique
    ),
}

for check_name, check_passed in primary_download_manifest_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(primary_download_manifest_checks.values()):
    raise ValueError(
        "Primary mutation download manifest validation failed."
    )

print(
    "\nPrimary download manifest rows: "
    f"{len(primary_download_manifest):,}"
)

print(
    "Expected payload size: "
    f"{primary_download_manifest['size'].sum() / (1024**3):.3f} GiB"
)

row_count_matches_handoff: True
file_ids_match_handoff: True
file_ids_are_unique: True

Primary download manifest rows: 9,426
Expected payload size: 0.649 GiB


In [35]:
# =============================================================================
# Define notebook 108 output paths
# =============================================================================

PRIMARY_DOWNLOAD_MANIFEST_PATH = (
    MUTATION_MANIFEST_DIR
    / (
        "gdc_manifest_tcga_frozen_multiomic_exact_sample_"
        "wxs_masked_somatic_mutation.txt"
    )
)

PRIMARY_FILE_HANDOFF_PATH = (
    Paths.genomics
    / "108_tcga_somatic_mutation_primary_file_handoff.csv"
)

CASE_ELIGIBILITY_PATH = (
    Paths.metadata
    / "108_tcga_somatic_mutation_case_eligibility.csv"
)

print("Notebook 108 output paths:")
print(
    "Primary download manifest: "
    f"{project_relative_path(PRIMARY_DOWNLOAD_MANIFEST_PATH)}"
)
print(
    "Primary file handoff:      "
    f"{project_relative_path(PRIMARY_FILE_HANDOFF_PATH)}"
)
print(
    "Case eligibility:          "
    f"{project_relative_path(CASE_ELIGIBILITY_PATH)}"
)

Notebook 108 output paths:
Primary download manifest: config/manifests/tcga_mutation/gdc_manifest_tcga_frozen_multiomic_exact_sample_wxs_masked_somatic_mutation.txt
Primary file handoff:      data/interim/genomics/108_tcga_somatic_mutation_primary_file_handoff.csv
Case eligibility:          data/interim/metadata/108_tcga_somatic_mutation_case_eligibility.csv


In [36]:
# =============================================================================
# Persist primary mutation acquisition artifacts
# =============================================================================

PRIMARY_DOWNLOAD_MANIFEST_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

PRIMARY_FILE_HANDOFF_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

CASE_ELIGIBILITY_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

primary_download_manifest = (
    primary_download_manifest
    .sort_values("id", kind="stable")
    .reset_index(drop=True)
)

primary_mutation_file_handoff = (
    primary_mutation_file_handoff
    .sort_values(
        [
            "tcga_case_barcode",
            "file_id",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

case_mutation_status = (
    case_mutation_status
    .sort_values(
        [
            "project_id",
            "tcga_case_barcode",
        ],
        kind="stable",
    )
    .reset_index(drop=True)
)

primary_download_manifest.to_csv(
    PRIMARY_DOWNLOAD_MANIFEST_PATH,
    sep="\t",
    index=False,
)

primary_mutation_file_handoff.to_csv(
    PRIMARY_FILE_HANDOFF_PATH,
    index=False,
)

case_mutation_status.to_csv(
    CASE_ELIGIBILITY_PATH,
    index=False,
)

print("Primary mutation acquisition artifacts written.")
print(
    f"Download manifest: {len(primary_download_manifest):,} rows"
)
print(
    f"File handoff:      {len(primary_mutation_file_handoff):,} rows"
)
print(
    f"Case eligibility:  {len(case_mutation_status):,} rows"
)

Primary mutation acquisition artifacts written.
Download manifest: 9,426 rows
File handoff:      9,426 rows
Case eligibility:  9,965 rows


In [37]:
# =============================================================================
# Validate persisted mutation acquisition artifacts
# =============================================================================

persisted_download_manifest = pd.read_csv(
    PRIMARY_DOWNLOAD_MANIFEST_PATH,
    sep="\t",
)

persisted_file_handoff = pd.read_csv(
    PRIMARY_FILE_HANDOFF_PATH,
)

persisted_case_eligibility = pd.read_csv(
    CASE_ELIGIBILITY_PATH,
)

roundtrip_checks = {
    "manifest_roundtrip_rows": (
        len(persisted_download_manifest)
        == len(primary_download_manifest)
    ),
    "handoff_roundtrip_rows": (
        len(persisted_file_handoff)
        == len(primary_mutation_file_handoff)
    ),
    "eligibility_roundtrip_rows": (
        len(persisted_case_eligibility)
        == len(case_mutation_status)
    ),
    "manifest_file_ids_preserved": (
        persisted_download_manifest["id"].tolist()
        == primary_download_manifest["id"].tolist()
    ),
    "handoff_file_ids_preserved": (
        persisted_file_handoff["file_id"].tolist()
        == primary_mutation_file_handoff["file_id"].tolist()
    ),
    "eligibility_case_ids_preserved": (
        persisted_case_eligibility["tcga_case_barcode"].tolist()
        == case_mutation_status["tcga_case_barcode"].tolist()
    ),
}

for check_name, check_passed in roundtrip_checks.items():
    print(f"{check_name}: {check_passed}")

if not all(roundtrip_checks.values()):
    raise ValueError(
        "Persisted mutation acquisition artifact validation failed."
    )

manifest_roundtrip_rows: True
handoff_roundtrip_rows: True
eligibility_roundtrip_rows: True
manifest_file_ids_preserved: True
handoff_file_ids_preserved: True
eligibility_case_ids_preserved: True


In [38]:
# =============================================================================
# Define primary mutation download directory
# =============================================================================

MUTATION_DOWNLOAD_DIR = (
    Paths.tcga
    / "somatic_mutation"
)

MUTATION_DOWNLOAD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "Mutation download directory: "
    f"{project_relative_path(MUTATION_DOWNLOAD_DIR)}"
)

Mutation download directory: data/raw/tcga/somatic_mutation


In [39]:
# =============================================================================
# Prepare GDC primary mutation download command
# =============================================================================

GDC_CLIENT_EXECUTABLE = "gdc-client"

download_manifest_path = project_relative_path(
    PRIMARY_DOWNLOAD_MANIFEST_PATH
)

download_directory_path = project_relative_path(
    MUTATION_DOWNLOAD_DIR
)

gdc_download_command = (
    f'{GDC_CLIENT_EXECUTABLE} download '
    f'-m "{download_manifest_path}" '
    f'-d "{download_directory_path}"'
)

print("GDC download command:")
print(gdc_download_command)

GDC download command:
gdc-client download -m "config/manifests/tcga_mutation/gdc_manifest_tcga_frozen_multiomic_exact_sample_wxs_masked_somatic_mutation.txt" -d "data/raw/tcga/somatic_mutation"


## Acquisition audit checkpoint

The TCGA/GDC somatic-mutation acquisition scope and primary Phase 4B handoff
have been prespecified and audited before payload download.

The authoritative GDC snapshot contains 10,171 open-access WXS
`Masked Somatic Mutation` MAF files generated by the
`Aliquot Ensemble Somatic Variant Merging and Masking` workflow.

Compatibility with the frozen 9,965-case TCGA multi-omic cohort was evaluated
using exact TCGA tumor-sample identity.

The resulting primary mutation-resource handoff contains:

- 9,192 frozen TCGA cases with at least one exact-sample mutation MAF;
- 9,426 eligible MAF files;
- 8,977 cases represented by one eligible MAF;
- 215 cases represented by multiple eligible MAFs from the same frozen tumor
  sample;
- 38 cases with mutation data available only from a different tumor vial,
  retained for audit but excluded from the primary handoff; and
- 735 frozen cases without an eligible mutation resource in the selected GDC
  snapshot.

Multiple eligible MAFs are retained without arbitrary file selection. No
mutation-content information was used to determine eligibility or resolve
biospecimen multiplicity.

The derived 9,426-file operational GDC manifest and the corresponding
file-level and case-level handoff tables have been persisted and validated by
round-trip reconstruction.

### Pending acquisition step

Raw MAF payload download and byte-level validation remain pending.

Notebook 108 is therefore at a reproducible **pre-download acquisition
checkpoint**, not yet analytically closed or frozen. Final closure requires:

1. download of the 9,426 manifest-defined MAF payloads;
2. validation of file presence, size, and MD5 against the derived manifest;
3. minimal MAF readability/schema inspection; and
4. final provenance metadata and registry publication.

No downstream mutation characterization should consume the resource until
these acquisition-validation steps are complete.